# Data Exploration: Sparkov Fraud Detection Dataset

This notebook explores the Sparkov synthetic fraud detection dataset, which simulates
realistic credit card transactions with labeled fraud indicators. The dataset contains
transaction metadata (amount, merchant, category, location) alongside cardholder
demographics and fraud labels.

**Goals of this notebook:**

1. Understand the structure and scale of the dataset.
2. Visualize transaction volume patterns and fraud distribution.
3. Engineer features suitable for ML-based fraud scoring.
4. Demonstrate how raw transaction records are converted into natural-language
   text strings for downstream embedding generation.

In [ ]:
import sys
import os

# Add project root to path so we can import src modules
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data.loader import SparkovDataLoader
from src.visualization.plots import (
    plot_transaction_volume_over_time,
    plot_amount_distribution,
    plot_category_fraud_rate,
)

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

print("Libraries loaded successfully.")

In [ ]:
# Load the Sparkov dataset
loader = SparkovDataLoader(data_path="../data/fraudTrain.csv")
df = loader.load()

print(f"Dataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print()
df.head()

In [ ]:
# Basic dataset statistics
print(f"Total transactions: {len(df):,}")
print(f"Fraud transactions: {df['is_fraud'].sum():,} ({df['is_fraud'].mean()*100:.2f}%)")
print(f"Legitimate transactions: {(~df['is_fraud'].astype(bool)).sum():,}")
print(f"Unique cardholders (cc_num): {df['cc_num'].nunique():,}")
print(f"Unique merchants: {df['merchant'].nunique():,}")
print(f"Merchant categories: {df['category'].nunique()}")
print()
df.info()

## Transaction Volume and Fraud Patterns

We begin by examining how transaction volume and fraud incidence vary over time,
across amount ranges, and by merchant category. These patterns inform both feature
engineering and the design of drift detection windows.

In [ ]:
# Parse the transaction timestamp into a datetime column
df["trans_datetime"] = pd.to_datetime(df["trans_date_trans_time"])

# Transaction volume over time (daily aggregation)
fig = plot_transaction_volume_over_time(
    df,
    datetime_col="trans_datetime",
    fraud_col="is_fraud",
    freq="D",
)
plt.title("Daily Transaction Volume: Legitimate vs Fraud")
plt.tight_layout()
plt.show()

In [ ]:
# Amount distribution by fraud label
fig = plot_amount_distribution(
    df,
    amount_col="amt",
    fraud_col="is_fraud",
)
plt.title("Transaction Amount Distribution: Fraud vs Legitimate")
plt.tight_layout()
plt.show()

# Summary statistics for amount by fraud label
print(df.groupby("is_fraud")["amt"].describe().T)

In [ ]:
# Fraud rate by merchant category
fig = plot_category_fraud_rate(
    df,
    category_col="category",
    fraud_col="is_fraud",
)
plt.title("Fraud Rate by Merchant Category")
plt.tight_layout()
plt.show()

## Feature Engineering for ML Model

Before training a fraud detection model, we extract numeric features from the raw
transaction data. Key derived features include:

- **Temporal features:** hour of day, day of week (fraud patterns are time-dependent).
- **Amount features:** raw amount, log-transformed amount.
- **Geographic features:** distance between cardholder and merchant coordinates.
- **Demographic proxies:** city population, cardholder age derived from date of birth.

We then inspect correlations among these features to check for multicollinearity
and to identify the most fraud-predictive signals.

In [ ]:
# Vectorized Haversine distance (returns kilometers)
lat1, lon1 = np.radians(df["lat"]), np.radians(df["long"])
lat2, lon2 = np.radians(df["merch_lat"]), np.radians(df["merch_long"])

dlat = lat2 - lat1
dlon = lon2 - lon1

a = np.sin(dlat / 2.0)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0)**2
c = 2 * np.arcsin(np.sqrt(a))
df["geo_distance"] = 6371.0 * c  # Earth radius in km

print(f"Geo distance stats (km):")
print(df["geo_distance"].describe())


## Transaction Text Generation for Embeddings

Our fraud detection pipeline converts each transaction record into a structured
natural-language string. This text representation is then passed to an embedding
model (sentence-transformers all-MiniLM-L6-v2) to produce a dense vector that captures
semantic relationships between transactions.

Below we demonstrate the `to_transaction_text` function on three sample transactions
to illustrate what the embedding model will receive as input.

In [ ]:
def to_transaction_text(row: pd.Series) -> str:
    """Convert a single Sparkov DataFrame row into a natural-language
    transaction description suitable for embedding generation.
    """
    text = (
        f"Transaction on {row['trans_date_trans_time']} "
        f"for ${row['amt']:.2f} at merchant {row['merchant']} "
        f"in category {row['category']}. "
        f"Cardholder: {row['first']} {row['last']}, "
        f"gender {row['gender']}, "
        f"located in {row['city']}, {row['state']} (zip {row['zip']}). "
        f"City population: {row['city_pop']:,}. "
        f"Job: {row['job']}. "
        f"Merchant location: ({row['merch_lat']:.4f}, {row['merch_long']:.4f})."
    )
    return text


# Show text representations for 3 example transactions
sample_indices = [0, 1, 2]
for idx in sample_indices:
    row = df.iloc[idx]
    text = to_transaction_text(row)
    fraud_label = "FRAUD" if row["is_fraud"] == 1 else "LEGITIMATE"
    print(f"--- Transaction {idx} [{fraud_label}] ---")
    print(text)
    print(f"Text length: {len(text)} characters")
    print()

## Summary of Key Findings

1. **Class imbalance.** The dataset exhibits a strong class imbalance typical of
   real-world fraud detection, with fraudulent transactions representing a small
   percentage of overall volume.

2. **Amount signal.** Fraudulent transactions tend to have higher amounts than
   legitimate ones, making `amt` (and its log transform) a useful feature.

3. **Category variation.** Fraud rates differ substantially across merchant
   categories, suggesting that category-specific thresholds or stratified
   models could improve detection.

4. **Temporal patterns.** Transaction volume and fraud incidence vary by hour
   and day of week, motivating time-based feature engineering.

5. **Text representation.** Each transaction can be serialized into a structured
   text string of approximately 200-400 characters, suitable for embedding
   with modern language models.

In the next notebook, we will generate embeddings from these text representations
and analyze the resulting embedding space for fraud vs. legitimate separation.